<a href="https://colab.research.google.com/github/ShamirAli55/flyrank-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShamirAli55/flyrank-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
%pip -q install duckdb huggingface_hub

import os
import getpass
import duckdb
import pandas as pd

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass("HF Token: ")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
TYPE huggingface,
TOKEN '{HF_TOKEN}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')"
}

## 1. Unit of analysis + time window


One row represents the daily search performance of one content page. For this assignment I will analyze data from **March 2026**, which is a mid-panel month and avoids using the final outcome period. This allows features to be created from historical observations without introducing future information.

In [9]:
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
LIMIT 5
""").df()

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727


## 2. Fields: feature / label / context / excluded

**Features** (what goes into the model — available before we know the outcome):
- `gsc_impressions` — how many times the page showed up in search
- `gsc_clicks` — raw click count from GSC
- `gsc_avg_position` — average SERP position over the month
- `content_visible_query_count` — number of distinct queries driving impressions
- `rare_impressions_share`, `anonymized_impressions_share` — query mix signals from the 90-day table

**Label** (what we're trying to predict):
- `is_declining` — a derived flag: 1 if performance dropped in the outcome window, 0 otherwise. It's a proxy based on a rule, not a manually reviewed ground truth.

**Context** (identifies the row, not a feature):
- `client_hash_id`, `content_hash_id`, `report_date` — used for grouping and splitting, dropped before training

**Excluded** (leaks the label — cannot use as features):
- `trend_pct`, `trend_direction` — these columns encode the outcome directly. Using them would give the model the answer during training.

In [10]:
con.sql(f"""
SELECT *
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
LIMIT 5
""").df()


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## Verification Queries

The following queries verify the unit of analysis, the available data for the selected month, and the completeness of the selected slice.

Query 1 — Verify the grain

In [11]:
grain = con.sql(f"""
SELECT
COUNT(*) AS total_rows,
COUNT(DISTINCT content_hash_id) AS unique_pages,
COUNT(DISTINCT report_date) AS unique_days
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
""").df()

grain

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_pages,unique_days
0,9841378,331437,31


Query 2 — Date range

In [12]:
date_range = con.sql(f"""
SELECT
MIN(report_date) AS start_date,
MAX(report_date) AS end_date
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
""").df()

date_range

,start_date,end_date
0,2026-03-01,2026-03-31


Query 3 — Availability (IS TRUE)

In [13]:
availability = con.sql(f"""
SELECT
COUNT(*) AS available_rows
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
AND gsc_data_available IS TRUE
""").df()

availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,3611061


## 4. Data limits

Things this dataset can't tell me:

- **GA4 data is sparse.** Many clients don't have `ga4_data_available = TRUE`, so engagement signals (sessions, scroll events) are only usable for a subset. I'm sticking to GSC features for now to keep the model applicable across all clients.
- **GSC rows can also be missing.** Not every content/date combination has search data — the query below shows what fraction is actually usable.
- **No content-level signals.** There's nothing about what the page actually says. A page could be declining because the topic went out of fashion, because of a competitor, or because of a technical issue — the data looks the same either way.
- **Client histories vary in length.** Some clients have been in the warehouse longer than others. Aggregating over all history without a fixed window would let longer-history clients dominate.

In [14]:
avail = con.sql(f"""
SELECT
    SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_rows,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_rows,
    COUNT(1) AS total_rows
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
""").df()

avail['gsc_pct'] = (avail['gsc_rows'] / avail['total_rows'] * 100).round(1)
avail['ga4_pct'] = (avail['ga4_rows'] / avail['total_rows'] * 100).round(1)
avail


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gsc_rows,ga4_rows,total_rows,gsc_pct,ga4_pct
0,3611061.0,413966.0,9841378,36.7,4.2


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.